# Tahap 2: Case Representation
**SubCPMK-3 — Penalaran Komputer | NIM: 202310370311358**

Tujuan: Mengekstrak metadata dan fitur teks dari setiap putusan, menyimpan dalam format terstruktur `.csv`.

In [ ]:
import re
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

txt_files = sorted(RAW_DIR.glob('*.txt'))
print(f'Dokumen tersedia: {len(txt_files)}')

## 2.1 Ekstraksi Metadata

In [ ]:
def extract_no_perkara(text):
    """Ekstrak nomor perkara (contoh: 123/Pid.Sus/2023/PN.Mks)"""
    pattern = r'(?:Nomor|No\.?)\s*:?\s*([\d]+/[A-Za-z.]+/\d{4}(?:/[A-Za-z.]+)*)'
    m = re.search(pattern, text, re.IGNORECASE)
    return m.group(1).strip() if m else 'UNKNOWN'

def extract_tanggal(text):
    """Ekstrak tanggal putusan."""
    months = r'(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)'
    pattern = rf'(\d{{1,2}})\s+{months}\s+(\d{{4}})'
    m = re.search(pattern, text, re.IGNORECASE)
    if m:
        return f'{m.group(1)} {m.group(2)} {m.group(3)}'
    return 'UNKNOWN'

def extract_pasal(text):
    """Ekstrak pasal-pasal yang disebut dalam putusan."""
    pattern = r'[Pp]asal\s+(\d+(?:\s*(?:ayat|[Aa]yat)?\s*(?:\(\d+\))?)?(?:\s+(?:jo\.?|juncto|dan)\s+[Pp]asal\s+\d+)*)'
    matches = re.findall(pattern, text)
    # Ambil 5 pasal unik pertama
    unique = list(dict.fromkeys([m.strip() for m in matches if m.strip()]))
    return '; '.join(unique[:5]) if unique else 'UNKNOWN'

def extract_pihak(text):
    """Ekstrak nama terdakwa."""
    pattern = r'(?:Terdakwa|TERDAKWA)\s*:?\s*([A-Z][A-Za-z\s]+?)(?:\n|,|;|berumur|Umur)'
    m = re.search(pattern, text)
    return m.group(1).strip() if m else 'UNKNOWN'

def extract_amar(text):
    """Ekstrak amar putusan (label solusi)."""
    # Cari section MENGADILI atau AMAR PUTUSAN
    pattern = r'(?:M E N G A D I L I|MENGADILI|AMAR PUTUSAN)(.{50,800})(?:Demikian|Ditetapkan|KETUA)'
    m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    if m:
        amar = m.group(1).strip()
        amar = re.sub(r'\s+', ' ', amar)
        return amar[:500]
    return 'UNKNOWN'

def classify_verdict(amar_text):
    """
    Klasifikasi amar putusan ke label:
    - 'bersalah_112' → Pasal 112 (kepemilikan)
    - 'bersalah_114' → Pasal 114 (peredaran)
    - 'bersalah_127' → Pasal 127 (pengguna)
    - 'bersalah_lain' → pasal lain
    - 'bebas' → dibebaskan
    """
    text_lower = amar_text.lower()
    if any(w in text_lower for w in ['bebas', 'tidak terbukti']):
        return 'bebas'
    if 'pasal 112' in text_lower:
        return 'bersalah_112'
    if 'pasal 114' in text_lower:
        return 'bersalah_114'
    if 'pasal 127' in text_lower:
        return 'bersalah_127'
    if any(w in text_lower for w in ['bersalah', 'terbukti', 'pidana']):
        return 'bersalah_lain'
    return 'unknown'

print('Fungsi ekstraksi metadata siap.')

## 2.2 Ekstraksi Konten Kunci & Feature Engineering

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

STOPWORDS_ID = set(stopwords.words('indonesian'))

def extract_ringkasan(text, n_sentences=3):
    """Ambil n_sentences kalimat pertama sebagai ringkasan fakta."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    ringkasan = ' '.join(sentences[:n_sentences])
    return ringkasan[:600]

def clean_for_model(text):
    """Preprocessing teks untuk model ML: lowercase, hapus punctuation & stopwords."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS_ID and len(t) > 2]
    return ' '.join(tokens)

def word_count(text):
    return len(text.split())

print('Fungsi feature engineering siap.')

## 2.3 Proses Semua Dokumen

In [ ]:
records = []

for f in tqdm(txt_files, desc='Memproses dokumen'):
    text = f.read_text(encoding='utf-8')
    case_id = f.stem

    no_perkara   = extract_no_perkara(text)
    tanggal      = extract_tanggal(text)
    pasal        = extract_pasal(text)
    pihak        = extract_pihak(text)
    amar         = extract_amar(text)
    label        = classify_verdict(amar)
    ringkasan    = extract_ringkasan(text)
    text_clean   = clean_for_model(text)
    jml_kata     = word_count(text)

    records.append({
        'case_id'       : case_id,
        'no_perkara'    : no_perkara,
        'tanggal'       : tanggal,
        'jenis_perkara' : 'Pidana Khusus Narkotika',
        'pasal'         : pasal,
        'pihak'         : pihak,
        'amar_putusan'  : amar,
        'label'         : label,
        'ringkasan_fakta': ringkasan,
        'text_clean'    : text_clean,
        'text_full'     : text,
        'word_count'    : jml_kata,
    })

df = pd.DataFrame(records)
print(f'\nTotal kasus diproses: {len(df)}')
print(f'Distribusi label:\n{df["label"].value_counts()}')

## 2.4 Simpan ke CSV & JSON

In [ ]:
# Simpan CSV (tanpa text_full untuk efisiensi)
cols_csv = ['case_id','no_perkara','tanggal','jenis_perkara','pasal',
            'pihak','amar_putusan','label','ringkasan_fakta','word_count']
df[cols_csv].to_csv(PROCESSED_DIR / 'cases.csv', index=False, encoding='utf-8-sig')

# Simpan JSON lengkap (untuk retrieval)
df.drop(columns=['text_full']).to_json(
    PROCESSED_DIR / 'cases.json', orient='records', force_ascii=False, indent=2
)

print(f'Saved: data/processed/cases.csv ({len(df)} baris)')
print(f'Saved: data/processed/cases.json')

# Preview
display(df[cols_csv].head(3))

In [ ]:
# Simpan text_clean terpisah untuk Tahap 3
df[['case_id','label','text_clean']].to_csv(
    PROCESSED_DIR / 'cases_clean.csv', index=False, encoding='utf-8-sig'
)
print('Saved: data/processed/cases_clean.csv')
print('\n>>> Tahap 2 SELESAI. Lanjut ke 03_retrieval.ipynb')